In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

ROOT = Path.cwd()
BATCH_DIR = ROOT / "results" / "raw" / "batch_500"
OLD_FILE = ROOT / "dataset" / "processed" / "all_monitoring.csv"
OUT_DIR = ROOT / "dataset" / "processed"
MODEL_DIR = ROOT / "ml"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Collect the existing 440 batch records
# --------------------------------------------------
frames = []

for file in sorted(BATCH_DIR.glob("*/monitoring_data.csv")):
    df = pd.read_csv(file)
    scenario = file.parent.name
    traffic = scenario.split("_")[-1]

    if traffic not in {"low", "medium", "high"}:
        print("Skipping unknown scenario:", scenario)
        continue

    df["traffic_level"] = traffic
    df["source_file"] = str(file.relative_to(ROOT))
    df["scenario"] = scenario
    frames.append(df)

if not frames:
    raise FileNotFoundError(
        f"No batch files found under {BATCH_DIR}"
    )

batch = pd.concat(frames, ignore_index=True)

# --------------------------------------------------
# 2. Add the existing 60 monitoring records
# --------------------------------------------------
if not OLD_FILE.exists():
    raise FileNotFoundError(f"Missing file: {OLD_FILE}")

old = pd.read_csv(OLD_FILE)
old["source_file"] = str(OLD_FILE.relative_to(ROOT))
old["scenario"] = "existing_processed_dataset"

combined = pd.concat([batch, old], ignore_index=True, sort=False)

raw_out = OUT_DIR / "monitoring_combined_500.csv"
combined.to_csv(raw_out, index=False)

print("BATCH FILES:", len(frames))
print("BATCH ROWS:", len(batch))
print("EXISTING ROWS:", len(old))
print("COMBINED ROWS:", len(combined))
print("Saved:", raw_out)

# --------------------------------------------------
# 3. Clean invalid records without deleting
#    potentially useful repeated simulation samples
# --------------------------------------------------
required = [
    "time",
    "total_queue_packets",
    "average_queue_packets",
    "utilization_percent",
    "traffic_level"
]

missing_cols = [c for c in required if c not in combined.columns]
if missing_cols:
    raise ValueError(f"Required columns missing: {missing_cols}")

clean = combined.copy()

numeric_cols = [
    "time",
    "total_queue_packets",
    "average_queue_packets",
    "utilization_percent"
]

for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

before = len(clean)

clean = clean.dropna(subset=required)
clean = clean[np.isfinite(clean[numeric_cols]).all(axis=1)]
clean = clean[clean["time"] >= 0]
clean = clean[clean["total_queue_packets"] >= 0]
clean = clean[clean["average_queue_packets"] >= 0]
clean = clean[
    clean["utilization_percent"].between(0, 100)
]
clean = clean[
    clean["traffic_level"].isin(["low", "medium", "high"])
].copy()

clean_out = OUT_DIR / "monitoring_filtered.csv"
clean.to_csv(clean_out, index=False)

print("\n--- CLEANING REPORT ---")
print("Rows before cleaning:", before)
print("Rows after cleaning:", len(clean))
print("Rows removed:", before - len(clean))
print("Saved filtered dataset:", clean_out)

# --------------------------------------------------
# 4. Data quality and duplicate audit
# --------------------------------------------------
print("\n--- TRAFFIC COUNTS ---")
print(clean["traffic_level"].value_counts().to_string())

print("\n--- MISSING VALUES ---")
print(clean[required].isnull().sum().to_string())

print("\n--- DUPLICATE MEASUREMENTS ---")
measurement_cols = required
print("Exact duplicate measurements:",
      int(clean.duplicated(subset=measurement_cols).sum()))
print("Duplicates are reported, not automatically deleted.")

print("\n--- FEATURE VARIATION ---")
print(clean[numeric_cols].describe().to_string())
print("\nUnique values per feature:")
print(clean[numeric_cols].nunique().to_string())

# --------------------------------------------------
# 5. Train only if real routing labels and
#    queue_A / queue_B features are present
# --------------------------------------------------
target = "selected_path"
features = ["queue_A", "queue_B"]

if target not in clean.columns:
    print("\nMODEL TRAINING NOT RUN.")
    print("Reason: the monitoring dataset has no selected_path label.")
    print("Do not invent labels or use the old model's predictions")
    print("as ground truth. Obtain actual route-decision labels first.")
elif not set(features).issubset(clean.columns):
    print("\nMODEL TRAINING NOT RUN.")
    print("Reason: queue_A and queue_B features are not both present.")
else:
    train_data = clean.dropna(subset=features + [target]).copy()
    train_data = train_data[
        np.isfinite(train_data[features]).all(axis=1)
    ]

    if train_data[target].nunique() < 2:
        print("\nMODEL TRAINING NOT RUN: fewer than two route classes.")
    elif train_data[target].value_counts().min() < 2:
        print("\nMODEL TRAINING NOT RUN: too few examples in a route class.")
    else:
        X = train_data[features]
        y = train_data[target]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y
        )

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        model_path = MODEL_DIR / "route_selector_retrained.pkl"
        joblib.dump(model, model_path)

        print("\n--- NEW MODEL EVALUATION ---")
        print("Test accuracy:", accuracy_score(y_test, predictions))
        print(classification_report(y_test, predictions, zero_division=0))
        print("New model saved:", model_path)
        print("The existing model has not been overwritten.")


BATCH FILES: 22
BATCH ROWS: 440
EXISTING ROWS: 60
COMBINED ROWS: 500
Saved: C:\CN\AI-Adaptive-Routing\dataset\processed\monitoring_combined_500.csv

--- CLEANING REPORT ---
Rows before cleaning: 500
Rows after cleaning: 500
Rows removed: 0
Saved filtered dataset: C:\CN\AI-Adaptive-Routing\dataset\processed\monitoring_filtered.csv

--- TRAFFIC COUNTS ---
traffic_level
high      180
medium    160
low       160

--- MISSING VALUES ---
time                     0
total_queue_packets      0
average_queue_packets    0
utilization_percent      0
traffic_level            0

--- DUPLICATE MEASUREMENTS ---
Exact duplicate measurements: 402
Duplicates are reported, not automatically deleted.

--- FEATURE VARIATION ---
             time  total_queue_packets  average_queue_packets  utilization_percent
count  500.000000           500.000000             500.000000                500.0
mean    11.500000            84.208000               2.339116                 10.0
std      5.772056           105.121

## routing logs se labels banana

## Actual routing dataset banao aur naya model train

In [6]:
import pandas as pd
from pathlib import Path

source = Path("dataset/processed/routing_training_data.csv")
output = Path("dataset/processed/routing_training_data_deduplicated.csv")

df = pd.read_csv(source)

print("Original shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())

# Remove only exact duplicate rows
clean_df = df.drop_duplicates().copy()

clean_df.to_csv(output, index=False)

print("\nCleaned shape:", clean_df.shape)
print("Rows removed:", len(df) - len(clean_df))
print("Saved to:", output.resolve())

if "selected_path" in clean_df.columns:
    print("\nTarget distribution:\n", clean_df["selected_path"].value_counts())

if "run_id" in clean_df.columns:
    print("\nSimulation runs:", clean_df["run_id"].nunique())

Original shape: (110, 9)

Columns: ['time', 'queue_A', 'queue_B', 'selected_path', 'total_queue_packets', 'average_queue_packets', 'utilization_percent', 'run_id', 'traffic_level']

Missing values:
 time                     0
queue_A                  0
queue_B                  0
selected_path            0
total_queue_packets      0
average_queue_packets    0
utilization_percent      0
run_id                   0
traffic_level            0
dtype: int64

Exact duplicate rows: 0

Cleaned shape: (110, 9)
Rows removed: 0
Saved to: C:\CN\AI-Adaptive-Routing\dataset\processed\routing_training_data_deduplicated.csv

Target distribution:
 selected_path
Path_A    73
Path_B    37
Name: count, dtype: int64

Simulation runs: 22


# Feature engineering aur data quality check

In [7]:
import pandas as pd
from pathlib import Path

source = Path("dataset/processed/routing_training_data_deduplicated.csv")
output = Path("dataset/processed/routing_features.csv")

if not source.exists():
    raise FileNotFoundError(
        "Deduplicated dataset nahi mila. Pehle Step 1 wala cell run karo."
    )

df = pd.read_csv(source)

required = {"queue_A", "queue_B", "selected_path", "run_id"}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Required columns missing: {sorted(missing)}")

print("Input shape:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())
print("\nTarget labels:\n", df["selected_path"].value_counts())
print("\nRuns:", df["run_id"].nunique())

# Keep only rows with valid features and labels
df["queue_A"] = pd.to_numeric(df["queue_A"], errors="coerce")
df["queue_B"] = pd.to_numeric(df["queue_B"], errors="coerce")

df = df.dropna(subset=["queue_A", "queue_B", "selected_path", "run_id"])
df = df[
    df["selected_path"].isin(["Path_A", "Path_B"])
    & (df["queue_A"] >= 0)
    & (df["queue_B"] >= 0)
].copy()

# Feature engineering from queue measurements
df["queue_difference"] = df["queue_A"] - df["queue_B"]
df["queue_difference_abs"] = df["queue_difference"].abs()
df["queue_total"] = df["queue_A"] + df["queue_B"]

# Avoid division by zero
df["queue_ratio"] = (
    df["queue_A"] / df["queue_B"].replace(0, float("nan"))
).fillna(0)

df = df.drop_duplicates().reset_index(drop=True)
df.to_csv(output, index=False)

print("\n--- FEATURE ENGINEERING RESULTS ---")
print("Final shape:", df.shape)
print("Features:", [
    "queue_A",
    "queue_B",
    "queue_difference",
    "queue_difference_abs",
    "queue_total",
    "queue_ratio",
])
print("\nTarget distribution:\n", df["selected_path"].value_counts())
print("\nSaved:", output.resolve())


Input shape: (110, 9)
Missing values:
 time                     0
queue_A                  0
queue_B                  0
selected_path            0
total_queue_packets      0
average_queue_packets    0
utilization_percent      0
run_id                   0
traffic_level            0
dtype: int64

Exact duplicates: 0

Target labels:
 selected_path
Path_A    73
Path_B    37
Name: count, dtype: int64

Runs: 22

--- FEATURE ENGINEERING RESULTS ---
Final shape: (110, 13)
Features: ['queue_A', 'queue_B', 'queue_difference', 'queue_difference_abs', 'queue_total', 'queue_ratio']

Target distribution:
 selected_path
Path_A    73
Path_B    37
Name: count, dtype: int64

Saved: C:\CN\AI-Adaptive-Routing\dataset\processed\routing_features.csv


In [8]:

import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix
)

# PATHS
root = Path.cwd()
src = root / "dataset/processed/routing_training_data.csv"
out = root / "dataset/processed"
res = root / "results"
mdl = root / "ml"
for p in [out, res, mdl]:
    p.mkdir(parents=True, exist_ok=True)

if not src.exists():
    raise FileNotFoundError(f"Dataset not found: {src}")

# STEP 1 — REMOVE EXACT DUPLICATES
df = pd.read_csv(src)
original_rows = len(df)
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("\nSTEP 1 — DUPLICATES")
print("Original:", original_rows, "| Duplicates removed:", duplicates_removed)

# STEP 2 — DATA QUALITY
required = ["queue_A", "queue_B", "selected_path", "run_id"]
missing_cols = set(required) - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df["queue_A"] = pd.to_numeric(df["queue_A"], errors="coerce")
df["queue_B"] = pd.to_numeric(df["queue_B"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)

rows_before = len(df)
df = df.dropna(subset=required).copy()
df = df[
    (df["queue_A"] >= 0)
    & (df["queue_B"] >= 0)
    & df["selected_path"].isin(["Path_A", "Path_B"])
].copy()
invalid_removed = rows_before - len(df)
df["run_id"] = df["run_id"].astype(str)

print("\nSTEP 2 — DATA QUALITY")
print("Invalid/missing rows removed:", invalid_removed)
print("Remaining rows:", len(df))
print("Simulation runs:", df["run_id"].nunique())
print("Target counts:\n", df["selected_path"].value_counts())
print("Missing values:\n", df.isnull().sum())

if df["run_id"].nunique() < 5:
    raise ValueError("At least 5 simulation runs are needed.")

df.to_csv(out / "routing_training_clean.csv", index=False)

# STEP 3 — FEATURE ENGINEERING
df["queue_difference"] = df["queue_A"] - df["queue_B"]
df["queue_difference_abs"] = df["queue_difference"].abs()
df["queue_total"] = df["queue_A"] + df["queue_B"]
df["queue_imbalance"] = (
    df["queue_difference_abs"] /
    df["queue_total"].replace(0, np.nan)
).fillna(0)

basic = ["queue_A", "queue_B"]
engineered = basic + [
    "queue_difference", "queue_difference_abs",
    "queue_total", "queue_imbalance"
]

df.to_csv(out / "routing_features.csv", index=False)

print("\nSTEP 3 — FEATURE ENGINEERING")
print("Basic features:", basic)
print("Engineered features:", engineered)

# STEP 4 — SPLIT BY WHOLE SIMULATION RUNS
X = df[engineered]
y = df["selected_path"]
groups = df["run_id"]

splitter = GroupShuffleSplit(n_splits=100, test_size=0.25, random_state=42)
chosen = None

for tr, te in splitter.split(X, y, groups):
    if y.iloc[tr].nunique() == 2 and y.iloc[te].nunique() == 2:
        chosen = (tr, te)
        break

if chosen is None:
    raise ValueError("Could not find a valid group split containing both labels.")

tr, te = chosen
train = df.iloc[tr].copy()
test = df.iloc[te].copy()

assert set(train["run_id"]).isdisjoint(set(test["run_id"]))

X_train_basic, X_test_basic = train[basic], test[basic]
X_train_eng, X_test_eng = train[engineered], test[engineered]
y_train, y_test = train["selected_path"], test["selected_path"]

print("\nSTEP 4 — GROUP TRAIN/TEST SPLIT")
print("Train rows:", len(train), "| Test rows:", len(test))
print("Train runs:", train["run_id"].nunique())
print("Test runs:", test["run_id"].nunique())
print("Run overlap:", len(set(train["run_id"]) & set(test["run_id"])))

# STEP 5 — TRAIN MODELS
estimators = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Random Forest - basic": RandomForestClassifier(
        n_estimators=300, random_state=42,
        class_weight="balanced", min_samples_leaf=2
    ),
    "Random Forest - engineered": RandomForestClassifier(
        n_estimators=300, random_state=42,
        class_weight="balanced", min_samples_leaf=2
    )
}

estimators["Dummy baseline"].fit(X_train_basic, y_train)
estimators["Random Forest - basic"].fit(X_train_basic, y_train)
estimators["Random Forest - engineered"].fit(X_train_eng, y_train)

test_X = {
    "Dummy baseline": X_test_basic,
    "Random Forest - basic": X_test_basic,
    "Random Forest - engineered": X_test_eng
}

# STEP 6 — FULL EVALUATION
print("\nSTEP 6 — TEST METRICS")
metric_rows = []
labels = ["Path_A", "Path_B"]
all_predictions = {}

for name, model in estimators.items():
    pred = model.predict(test_X[name])
    all_predictions[name] = pred

    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, pred, average="weighted", zero_division=0)
    })

    print(f"\n--- {name} ---")
    print(classification_report(y_test, pred, labels=labels, zero_division=0))
    cm = confusion_matrix(y_test, pred, labels=labels)
    print("Confusion matrix (actual rows; predicted columns):\n", cm)

    safe = name.lower().replace(" ", "_").replace("-", "")
    pd.DataFrame(
        cm,
        index=["actual_Path_A", "actual_Path_B"],
        columns=["predicted_Path_A", "predicted_Path_B"]
    ).to_csv(res / f"confusion_matrix_{safe}.csv")

metrics = pd.DataFrame(metric_rows)
print("\nMODEL COMPARISON")
print(metrics.to_string(index=False))
metrics.to_csv(res / "routing_model_comparison.csv", index=False)

# Group cross-validation only on training runs
print("\nGROUP CROSS-VALIDATION (TRAINING DATA ONLY)")
cv_rows = []
n_splits = min(5, train["run_id"].nunique())

if n_splits >= 2:
    cv = GroupKFold(n_splits=n_splits)
    scoring = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

    for name, features in [
        ("Random Forest - basic", basic),
        ("Random Forest - engineered", engineered)
    ]:
        cv_model = RandomForestClassifier(
            n_estimators=300, random_state=42,
            class_weight="balanced", min_samples_leaf=2
        )
        scores = cross_validate(
            cv_model, train[features], y_train,
            groups=train["run_id"], cv=cv, scoring=scoring
        )
        row = {"model": name}
        for score_name in scoring:
            row[score_name + "_mean"] = scores["test_" + score_name].mean()
            row[score_name + "_std"] = scores["test_" + score_name].std()
        cv_rows.append(row)

    cv_df = pd.DataFrame(cv_rows)
    print(cv_df.to_string(index=False))
    cv_df.to_csv(res / "routing_group_cv_metrics.csv", index=False)

# Choose candidate by macro F1, then balanced accuracy
rf_names = ["Random Forest - basic", "Random Forest - engineered"]
best_name = max(
    rf_names,
    key=lambda name: (
        metrics.loc[metrics["model"] == name, "f1_macro"].iloc[0],
        metrics.loc[metrics["model"] == name, "balanced_accuracy"].iloc[0]
    )
)

best_model = estimators[best_name]
best_features = basic if best_name == "Random Forest - basic" else engineered

model_path = mdl / "route_selector_cleaned_candidate.pkl"
joblib.dump({
    "model": best_model,
    "features": best_features,
    "model_name": best_name
}, model_path)

summary = {
    "original_rows": original_rows,
    "exact_duplicates_removed": duplicates_removed,
    "invalid_rows_removed": invalid_removed,
    "clean_rows": len(df),
    "simulation_runs": int(df["run_id"].nunique()),
    "train_rows": len(train),
    "test_rows": len(test),
    "train_runs": int(train["run_id"].nunique()),
    "test_runs": int(test["run_id"].nunique()),
    "run_overlap": len(set(train["run_id"]) & set(test["run_id"])),
    "selected_candidate": best_name,
    "candidate_model": str(model_path),
    "metrics_file": str(res / "routing_model_comparison.csv")
}

summary_path = res / "routing_training_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\nALL 6 STEPS COMPLETED")
print("Selected candidate:", best_name)
print("Saved model:", model_path)
print("Saved summary:", summary_path)
print("Existing route_selector.pkl remains untouched.")



STEP 1 — DUPLICATES
Original: 110 | Duplicates removed: 0

STEP 2 — DATA QUALITY
Invalid/missing rows removed: 0
Remaining rows: 110
Simulation runs: 22
Target counts:
 selected_path
Path_A    73
Path_B    37
Name: count, dtype: int64
Missing values:
 time                     0
queue_A                  0
queue_B                  0
selected_path            0
total_queue_packets      0
average_queue_packets    0
utilization_percent      0
run_id                   0
traffic_level            0
dtype: int64

STEP 3 — FEATURE ENGINEERING
Basic features: ['queue_A', 'queue_B']
Engineered features: ['queue_A', 'queue_B', 'queue_difference', 'queue_difference_abs', 'queue_total', 'queue_imbalance']

STEP 4 — GROUP TRAIN/TEST SPLIT
Train rows: 80 | Test rows: 30
Train runs: 16
Test runs: 6
Run overlap: 0

STEP 6 — TEST METRICS

--- Dummy baseline ---
              precision    recall  f1-score   support

      Path_A       0.77      1.00      0.87        23
      Path_B       0.00      0.00    

## ML model ko NS-3 ke saath integrate karna.

In [9]:
from pathlib import Path
import joblib

model_path = Path("ml/route_selector_cleaned_candidate.pkl")
bundle = joblib.load(model_path)

print("Model:", bundle["model_name"])
print("Features:", bundle["features"])
print("Model file exists:", model_path.exists())
print("Feature count:", len(bundle["features"]))

Model: Random Forest - basic
Features: ['queue_A', 'queue_B']
Model file exists: True
Feature count: 2


In [10]:

from pathlib import Path
import hashlib
import pandas as pd

ROOT = Path(r"C:\CN\AI-Adaptive-Routing")

if not ROOT.exists():
    raise FileNotFoundError(f"Project folder not found: {ROOT}")

records = []

for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    # Ignore generated/cache folders during the first audit
    if any(part in {".git", "__pycache__", ".ipynb_checkpoints"}
           for part in p.parts):
        continue

    try:
        size = p.stat().st_size
        digest = None

        # Hash files to identify exact duplicates, without modifying them
        if size <= 100 * 1024 * 1024:
            h = hashlib.sha256()
            with p.open("rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    h.update(chunk)
            digest = h.hexdigest()

        records.append({
            "file": str(p.relative_to(ROOT)),
            "size_KB": round(size / 1024, 2),
            "extension": p.suffix.lower() or "(no extension)",
            "sha256": digest
        })
    except (PermissionError, OSError) as e:
        records.append({
            "file": str(p.relative_to(ROOT)),
            "size_KB": None,
            "extension": p.suffix.lower() or "(no extension)",
            "sha256": None
        })

audit = pd.DataFrame(records).sort_values("file")
print("Total files:", len(audit))

print("\nFILES BY FOLDER")
audit["top_folder"] = audit["file"].str.split("\\").str[0]
print(audit.groupby("top_folder").agg(
    files=("file", "count"),
    total_KB=("size_KB", "sum")
).sort_values("total_KB", ascending=False).to_string())

print("\nPOSSIBLE TEMPORARY / BACKUP FILES — REVIEW ONLY")
review = audit[
    audit["file"].str.lower().str.contains(
        r"\.bak$|\.tmp$|\.temp$|backup|old|copy|checkpoint|\.log$",
        regex=True
    )
]
print(review[["file", "size_KB"]].to_string(index=False)
      if not review.empty else "No obvious temporary/backup filenames found.")

print("\nEXACT DUPLICATE FILES — REVIEW ONLY")
hashed = audit.dropna(subset=["sha256"])
dupes = hashed[hashed.duplicated("sha256", keep=False)].sort_values("sha256")
print(dupes[["file", "size_KB"]].to_string(index=False)
      if not dupes.empty else "No exact duplicates found.")

# Save the audit report; no project files are deleted or changed.
report = ROOT / "results" / "project_file_audit.csv"
report.parent.mkdir(parents=True, exist_ok=True)
audit.drop(columns=["sha256", "top_folder"], errors="ignore").to_csv(
    report, index=False
)
print("\nAudit report saved:", report)


Total files: 244

FILES BY FOLDER
                                             files  total_KB
top_folder                                                  
results                                        200   5008.66
ml                                               6    454.73
ns3                                              4    128.74
dataset                                         19    122.38
01_Dataset_Analysis_and_ML.ipynb                 1    115.96
02_AI_Routing_Engine.ipynb                       1     49.98
build-error.log                                  1     43.32
03_NS3_Adaptive_Routing.ipynb                    1     40.03
update_AI_Routing_Engine.ipynb                   1     34.09
compiler_test.exe                                1     33.85
04_AI_Adaptive_Routing_Implementation.ipynb      1     32.89
05_ML_Routing_Integration.ipynb                  1     30.03
01_Dataset_Analysis.ipynb                        1     12.70
README.md                                        1 